<a href="https://colab.research.google.com/github/mehakknasirr/neurofive-ml-track/blob/main/ML_Pipeline_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Dataset Load
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# 2. Feature Engineering (2 New Features)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Features and Target selection
numeric_features = ['Age', 'Fare', 'FamilySize']
categorical_features = ['Sex', 'Embarked', 'Pclass']
X = df[numeric_features + categorical_features]
y = df['Survived']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Preprocessing Pipelines using ColumnTransformer
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, numeric_features),
    ('cat', cat_pipeline, categorical_features)
])

# 4. Full ML Pipeline (Preprocessor + Classifier)
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Fit and Evaluate
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)

print(f"✅ Pipeline Model Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%\n")
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Did Not Survive', 'Survived']))

# 5. Save the Pipeline Model using joblib
joblib.dump(full_pipeline, 'titanic_pipeline_model.pkl')
print("✅ Final pipeline saved successfully as 'titanic_pipeline_model.pkl'!")


✅ Pipeline Model Accuracy: 79.89%

--- Classification Report ---
                 precision    recall  f1-score   support

Did Not Survive       0.81      0.88      0.84       110
       Survived       0.78      0.67      0.72        69

       accuracy                           0.80       179
      macro avg       0.79      0.77      0.78       179
   weighted avg       0.80      0.80      0.80       179

✅ Final pipeline saved successfully as 'titanic_pipeline_model.pkl'!


### ⚙️ Why Use Scikit-Learn Pipelines?

1. **Prevents Data Leakage:** Ensures transformations (like scaling and imputation) are fitted ONLY on the training data and then applied to the test data.
2. **Clean & Reusable Code:** Chains preprocessing steps and model training into a single object (`full_pipeline.fit()` and `full_pipeline.predict()`).
3. **Feature Engineering Impact:** Added `FamilySize` and `IsAlone` features to better capture passenger survival dynamics.
4. **Export Ready:** Saved the entire trained pipeline using `joblib` so it can be deployed directly into production environments.
